In [ ]:
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

def install_nodeps(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", package])

install_nodeps("insightface")

install("prettytable")
install("easydict")
install("Cython")

install("onnxruntime-gpu")
install("faiss-cpu")
install("supervision")
install("lap")

install("transformers")
install("huggingface_hub")
install("safetensors")
install("timm")
install("omegaconf")
 
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "diffusers>=0.27.0"])
install("accelerate")

print("✅ All dependencies installed!")

In [ ]:
import os
import sys
import types
import cv2
import numpy as np
import faiss
import json
import time
import warnings
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

_fake_mask_renderer = types.ModuleType("insightface.app.mask_renderer")
sys.modules["insightface.app.mask_renderer"] = _fake_mask_renderer

import insightface
from insightface.app import FaceAnalysis
import supervision as sv

warnings.filterwarnings("ignore")

@dataclass
class Config:

    ENROLLMENT_DIR: str = "/kaggle/input/datasets/aadityas33011/vitdataset/vidData/solo-pics"
    VIDEO_DIR: str = "/kaggle/input/datasets/aadityas33011/vitdataset/vidData/video"
    OUTPUT_DIR: str = "/kaggle/working/output"
    
    DET_MODEL: str = "buffalo_l"
    DET_SIZE: Tuple[int, int] = (640, 640)
    DET_THRESH: float = 0.35
    
    EMBEDDING_DIM: int = 512
    
    MATCH_THRESHOLD_BASE: float = 0.28
    MATCH_THRESHOLD_LAMBDA: float = 0.10
    UNKNOWN_LABEL: str = "Unknown"
    
    TRACK_THRESH: float = 0.20
    TRACK_BUFFER: int = 60
    MATCH_THRESH: float = 0.7
    VOTE_WINDOW: int = 4
    VOTE_MIN: int = 2
    
    ENABLE_GGA: bool = True
    GGA_FALLBACK: bool = True
    GGA_PROMPTS: List[str] = field(default_factory=lambda: [
        "a photorealistic portrait of a person in harsh bright sunlight, highly detailed",
        "a photorealistic portrait of a person looking slightly to the side, natural expression",
        "a photorealistic portrait of a person in dim cinematic lighting, sharp focus",
        "a photorealistic portrait of a person looking slightly down, indoor office lighting",
    ])
    
    PROCESS_EVERY_N_FRAMES: int = 2
    RECOGNIZE_EVERY_N_FRAMES: int = 2
    MAX_FRAMES: int = 900
    
    BBOX_COLOR: Tuple[int, int, int] = (0, 255, 0)
    UNKNOWN_COLOR: Tuple[int, int, int] = (0, 0, 255)
    FONT_SCALE: float = 0.6
    FONT_THICKNESS: int = 2

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

print(f"📁 Enrollment dir: {config.ENROLLMENT_DIR}")
print(f"🎥 Video dir:      {config.VIDEO_DIR}")
print(f"📂 Output dir:     {config.OUTPUT_DIR}")
print(f"✅ Configuration loaded!")

In [ ]:
class FaceDetector:

    def __init__(self, config: Config):
        print("🔍 Loading SCRFD face detector...")
        self.app = FaceAnalysis(
            name=config.DET_MODEL,
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
        )
        self.app.prepare(ctx_id=0, det_size=config.DET_SIZE, det_thresh=config.DET_THRESH)
        self.config = config
        self.rec_model = None
        for name, model in self.app.models.items():
            taskname = getattr(model, 'taskname', '')
            if taskname == 'recognition':
                self.rec_model = model
                print(f"   ✅ Found recognition model: {name} (taskname={taskname})")
                break
        if self.rec_model is None:
            for name, model in self.app.models.items():
                input_size = getattr(model, 'input_size', None)
                if input_size and input_size == (112, 112):
                    self.rec_model = model
                    print(f"   ✅ Found recognition model by input size: {name}")
                    break
        print(f"   ✅ Detector ready (input size: {config.DET_SIZE})")
        print(f"   ✅ Recognition model extracted: {self.rec_model is not None}")
    
    def get_embedding_direct(self, aligned_face: np.ndarray) -> Optional[np.ndarray]:
        
        if self.rec_model is None:
            return None
        try:
            if aligned_face.shape[:2] != (112, 112):
                aligned_face = cv2.resize(aligned_face, (112, 112))
            if hasattr(self.rec_model, 'get_feat'):
                blob = cv2.dnn.blobFromImage(
                    aligned_face, 1.0/127.5, (112, 112), (127.5, 127.5, 127.5), swapRB=True
                )
                embedding = self.rec_model.session.run(
                    [self.rec_model.session.get_outputs()[0].name],
                    {self.rec_model.session.get_inputs()[0].name: blob}
                )[0][0]
                return embedding
            elif hasattr(self.rec_model, 'get'):
                from insightface.utils.face_align import norm_crop
                embedding = self.rec_model.get(aligned_face)
                return embedding
        except Exception as e:
            print(f"      ⚠️ Direct embedding failed: {e}")
        return None
    
    def detect(self, frame: np.ndarray) -> list:
        
        faces = self.app.get(frame)
        return faces
    
    def detect_for_enrollment(self, image: np.ndarray):
        
        faces = self.detect(image)
        if not faces:
            return None
        return max(faces, key=lambda f: f.det_score)
    
    def visualize(self, frame: np.ndarray, faces: list) -> np.ndarray:
        
        vis = frame.copy()
        for face in faces:
            bbox = face.bbox.astype(int)
            cv2.rectangle(vis, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 0), 2)
            if face.kps is not None:
                for kp in face.kps:
                    cv2.circle(vis, (int(kp[0]), int(kp[1])), 2, (0, 0, 255), -1)
        return vis

detector = FaceDetector(config)
print("✅ Face detector initialized!")

In [ ]:
class ViTFaceRecognizer:

    def __init__(self, config: Config):
        self.config = config
        self.model = None
        self.available = False
        self.device = "cpu"
        
        self._try_load_vit()
    
    def _try_load_vit(self):
        
        try:
            import torch
            from huggingface_hub import hf_hub_download
            
            print("🧠 Loading AdaFace ViT-Base (KP-RPE) from HuggingFace...")
            
            repo_id = "minchul/cvlface_adaface_vit_base_kprpe_webface12m"
            cache_path = os.path.expanduser("~/.cvlface_cache/adaface_vit")
            os.makedirs(cache_path, exist_ok=True)
            
            print("   ⏳ Downloading model files...")
            required_files = ['config.json', 'wrapper.py', 'model.safetensors']
            
            try:
                hf_hub_download(repo_id, 'files.txt', local_dir=cache_path, 
                               local_dir_use_symlinks=False)
                with open(os.path.join(cache_path, 'files.txt'), 'r') as f:
                    extra_files = [line.strip() for line in f.read().split('\n') if line.strip()]
                required_files = list(set(required_files + extra_files))
            except Exception:
                pass
            
            for fname in required_files:
                full_path = os.path.join(cache_path, fname)
                if not os.path.exists(full_path):
                    try:
                        hf_hub_download(repo_id, fname, local_dir=cache_path,
                                       local_dir_use_symlinks=False)
                    except Exception as e:
                        print(f"      ⚠️ Could not download {fname}: {e}")
            
            print("   🔧 Pre-loading RPE CUDA extension...")
            
            site_packages_dirs = [
                os.path.expanduser("~/.local/lib/python3.12/site-packages"),
                os.path.expanduser("~/.local/lib/python3.11/site-packages"),
                os.path.expanduser("~/.local/lib/python3.10/site-packages"),
            ]
            for sp_dir in site_packages_dirs:
                if os.path.exists(sp_dir):
                    for f in os.listdir(sp_dir):
                        if f.startswith("rpe_index") and f.endswith(".egg"):
                            egg_path = os.path.join(sp_dir, f)
                            if egg_path not in sys.path:
                                sys.path.insert(0, egg_path)
                                print(f"      ✅ Added RPE egg: {egg_path}")
            
            try:
                import rpe_index_cpp
                print("      ✅ rpe_index_cpp available")
            except ImportError:
                print("      ⚠️ rpe_index_cpp not compiled yet")
            
            _original_sys_exit = sys.exit
            sys.exit = lambda *a, **kw: None
            
            stale_patterns = ['rpe_index', 'RPE', 'vit_kprpe', 'kprpe', 
                              'models.vit_kprpe', 'models.base',
                              'transformers_modules']
            mods_to_clear = [k for k in list(sys.modules.keys()) 
                            if any(p in k for p in stale_patterns)]
            for mod in mods_to_clear:
                del sys.modules[mod]
            if mods_to_clear:
                print(f"      🧹 Cleared {len(mods_to_clear)} stale modules")
            
            print("   📦 Loading model directly (bypassing AutoModel)...")
            
            cwd = os.getcwd()
            os.chdir(cache_path)
            if cache_path not in sys.path:
                sys.path.insert(0, cache_path)
            
            try:
                import json
                with open(os.path.join(cache_path, 'config.json'), 'r') as f:
                    config_json = json.load(f)
                
                from omegaconf import OmegaConf
                model_conf = OmegaConf.create(config_json.get('conf', config_json))
                
                from models import get_model
                self.model = get_model(model_conf)
                
                weight_path = os.path.join(cache_path, 'pretrained_model', 'model.pt')
                if os.path.exists(weight_path):
                    print(f"      📥 Loading weights from model.pt...")
                    self.model.load_state_dict_from_path(weight_path)
                else:
                    safetensors_path = os.path.join(cache_path, 'model.safetensors')
                    if os.path.exists(safetensors_path):
                        print(f"      📥 Loading weights from model.safetensors...")
                        from safetensors.torch import load_file
                        state_dict = load_file(safetensors_path)
                        cleaned = {}
                        for k, v in state_dict.items():
                            clean_key = k.replace('model.', '', 1) if k.startswith('model.') else k
                            cleaned[clean_key] = v
                        self.model.load_state_dict(cleaned, strict=False)
                    else:
                        raise FileNotFoundError("No weight file found (model.pt or model.safetensors)")
                
                print("      ✅ Weights loaded!")
                
            finally:
                sys.exit = _original_sys_exit
                os.chdir(cwd)
            
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
            self.model = self.model.to(self.device)
            self.model.eval()
            
            self.available = True
            print(f"   ✅ AdaFace ViT loaded! (device: {self.device})")
            print(f"   📐 Architecture: ViT-Base + KeyPoint Relative Position Encoding")
            print(f"   📊 Training data: WebFace12M (12M faces)")
            
        except Exception as e:
            print(f"   ⚠️  AdaFace ViT failed to load: {e}")
            import traceback
            traceback.print_exc()
            print(f"   📸 Will use ArcFace ResNet-50 fallback for recognition.")
            self.available = False
    
    def get_embedding(self, face_chip: np.ndarray, keypoints: np.ndarray = None) -> Optional[np.ndarray]:
        
        if not self.available or self.model is None:
            return None
        
        try:
            import torch
            from torchvision.transforms import Compose, ToTensor, Normalize
            from PIL import Image
            
            face_rgb = cv2.cvtColor(face_chip, cv2.COLOR_BGR2RGB)
            if face_rgb.shape[:2] != (112, 112):
                face_rgb = cv2.resize(face_rgb, (112, 112))
            face_pil = Image.fromarray(face_rgb)
            
            transform = Compose([
                ToTensor(),
                Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
            input_tensor = transform(face_pil).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                std_keypoints = np.array([
                    [38.2946 / 112.0, 51.6963 / 112.0],
                    [73.5318 / 112.0, 51.5014 / 112.0],
                    [56.0252 / 112.0, 71.7366 / 112.0],
                    [41.5493 / 112.0, 92.3655 / 112.0],
                    [70.7299 / 112.0, 92.2041 / 112.0],
                ], dtype=np.float32)
                
                try:
                    kps_tensor = torch.tensor(
                        std_keypoints, dtype=torch.float32
                    ).unsqueeze(0).to(self.device)
                    embedding = self.model(input_tensor, kps_tensor)
                except (TypeError, RuntimeError):
                    embedding = self.model(input_tensor)
            
            if isinstance(embedding, (tuple, list)):
                embedding = embedding[0]
            
            emb_np = embedding.cpu().numpy().flatten()
            
            norm = np.linalg.norm(emb_np)
            if norm > 0:
                emb_np = emb_np / norm
            
            return emb_np.astype(np.float32)
            
        except Exception as e:
            print(f"      ⚠️ ViT embedding failed: {e}")
            return None

vit_recognizer = ViTFaceRecognizer(config)

if not hasattr(detector, '_original_unpatched_get_embedding_direct'):
    detector._original_unpatched_get_embedding_direct = detector.get_embedding_direct

def _vit_enhanced_get_embedding_direct(aligned_face: np.ndarray, keypoints: np.ndarray = None) -> Optional[np.ndarray]:
    
    if vit_recognizer.available:
        vit_emb = vit_recognizer.get_embedding(aligned_face, keypoints)
        if vit_emb is not None:
            return vit_emb
    
    return detector._original_unpatched_get_embedding_direct(aligned_face)

detector.get_embedding_direct = _vit_enhanced_get_embedding_direct
print("✅ ViT-enhanced recognition ready!" if vit_recognizer.available 
      else "✅ Using ArcFace ResNet-50 recognition (ViT unavailable)")

In [ ]:
class FaceAligner:

    REFERENCE_LANDMARKS = np.array([
        [38.2946, 51.6963],
        [73.5318, 51.5014],
        [56.0252, 71.7366],
        [41.5493, 92.3655],
        [70.7299, 92.2041],
    ], dtype=np.float32)
    
    def __init__(self, output_size: int = 112, apply_clahe: bool = True):
        self.output_size = output_size
        self.apply_clahe = apply_clahe
        if apply_clahe:
            self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        print(f"   ✅ Aligner ready (output: {output_size}×{output_size}, CLAHE: {apply_clahe})")
    
    def align(self, frame: np.ndarray, landmarks: np.ndarray) -> np.ndarray:
        
        M = self._estimate_similarity_transform(landmarks, self.REFERENCE_LANDMARKS)
        aligned = cv2.warpAffine(frame, M, (self.output_size, self.output_size))
        
        if self.apply_clahe:
            aligned = self._apply_clahe(aligned)
        
        return aligned
    
    def _estimate_similarity_transform(self, src: np.ndarray, dst: np.ndarray) -> np.ndarray:
        
        num = src.shape[0]
        dim = src.shape[1]
        
        src_mean = src.mean(axis=0)
        dst_mean = dst.mean(axis=0)
        
        src_demean = src - src_mean
        dst_demean = dst - dst_mean
        
        A = np.zeros((num * dim, 4))
        b = np.zeros(num * dim)
        
        for i in range(num):
            A[2*i]     = [src_demean[i, 0], -src_demean[i, 1], 1, 0]
            A[2*i + 1] = [src_demean[i, 1],  src_demean[i, 0], 0, 1]
            b[2*i]     = dst_demean[i, 0]
            b[2*i + 1] = dst_demean[i, 1]
        
        result, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        
        M = np.array([
            [result[0], -result[1], result[2] + dst_mean[0] - result[0]*src_mean[0] + result[1]*src_mean[1]],
            [result[1],  result[0], result[3] + dst_mean[1] - result[1]*src_mean[0] - result[0]*src_mean[1]]
        ])
        
        return M
    
    def _apply_clahe(self, image: np.ndarray) -> np.ndarray:
        
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l_channel, a_channel, b_channel = cv2.split(lab)
        l_channel = self.clahe.apply(l_channel)
        lab = cv2.merge([l_channel, a_channel, b_channel])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

aligner = FaceAligner(output_size=112, apply_clahe=True)
print("✅ Face aligner initialized!")

In [ ]:
class FaceRecognizer:

    def __init__(self, config: Config):
        self.config = config
        self.embedding_dim = config.EMBEDDING_DIM
        print(f"   ✅ Recognizer ready (dim: {self.embedding_dim})")``
    
    @staticmethod
    def normalize(embedding: np.ndarray) -> np.ndarray:
        
        norm = np.linalg.norm(embedding)
        if norm == 0:
            return embedding
        return embedding / norm
    
    @staticmethod
    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))
    
    def compute_quality_score(self, face) -> float:
        
        bbox = face.bbox
        face_width = bbox[2] - bbox[0]
        face_height = bbox[3] - bbox[1]
        face_area = face_width * face_height
        
        size_score = np.clip(face_area / (200 * 200), 0, 1)
        
        conf_score = float(face.det_score)
        
        quality = 0.5 * size_score + 0.5 * conf_score
        return float(np.clip(quality, 0, 1))
    
    def get_dynamic_threshold(self, quality_score: float) -> float:
        
        return self.config.MATCH_THRESHOLD_BASE + \
               self.config.MATCH_THRESHOLD_LAMBDA * (1.0 - quality_score)

recognizer = FaceRecognizer(config)
print("✅ Face recognizer initialized!")

In [ ]:
class TraditionalAugmentor:

    def __init__(self):
        print("   📸 Traditional augmentor initialized")
    
    def generate_variations(self, face_chip: np.ndarray) -> List[Tuple[str, np.ndarray]]:
        
        variations = []
        h, w = face_chip.shape[:2]
        
        M = cv2.getRotationMatrix2D((w//2, h//2), 10, 1.0)
        rotated_left = cv2.warpAffine(face_chip, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        variations.append(("rotated_left", rotated_left))
        
        M = cv2.getRotationMatrix2D((w//2, h//2), -10, 1.0)
        rotated_right = cv2.warpAffine(face_chip, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        variations.append(("rotated_right", rotated_right))
        
        darkened = cv2.convertScaleAbs(face_chip, alpha=0.6, beta=-20)
        variations.append(("darkened", darkened))
        
        brightened = cv2.convertScaleAbs(face_chip, alpha=1.3, beta=30)
        variations.append(("brightened", brightened))
        
        blurred = cv2.GaussianBlur(face_chip, (5, 5), 1.5)
        variations.append(("blurred", blurred))
        
        flipped = cv2.flip(face_chip, 1)
        variations.append(("flipped", flipped))
        
        lab = cv2.cvtColor(face_chip, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(4, 4))
        l = clahe.apply(l)
        enhanced = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
        variations.append(("contrast_enhanced", enhanced))
        
        occluded = face_chip.copy()
        occluded[h//2:, :] = cv2.GaussianBlur(occluded[h//2:, :], (21, 21), 10)
        variations.append(("lower_occluded", occluded))
        
        return variations

class GenerativeAugmentor:

    def __init__(self, config: Config):
        self.config = config
        self.pipeline = None
        self.available = False
        self.fallback = TraditionalAugmentor()
        
        if config.ENABLE_GGA:
            self._try_load_instantid()
    
    def _try_load_instantid(self):
        
        try:
            import torch
            from huggingface_hub import hf_hub_download
            from diffusers import ControlNetModel, DDIMScheduler
            import diffusers
            
            print("🎨 Attempting to load InstantID pipeline...")
            print(f"   📦 diffusers version: {diffusers.__version__}")
            print("   ⏳ This may take a few minutes on first run...")
            
            if torch.cuda.is_available():
                vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
                print(f"   GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)")
                
                if vram_gb < 14.0:
                    print("   ⚠️  Less than 14.0GB VRAM — InstantID may OOM.")
                    print("   📸 Using traditional augmentation as primary method.")
                    return
            
            print("   📥 Downloading InstantID weights...")
            controlnet = ControlNetModel.from_pretrained(
                "InstantX/InstantID",
                subfolder="ControlNetModel",
                torch_dtype=torch.float16,
            )
            
            face_adapter_path = hf_hub_download(
                "InstantX/InstantID",
                filename="ip-adapter.bin"
            )
            print(f"      ✅ IP-Adapter downloaded: {face_adapter_path}")
            
            try:
                import sys
                import os
                import urllib.request
                
                script_name = "pipeline_stable_diffusion_xl_instantid.py"
                if not os.path.exists(script_name):
                    print("   📥 Downloading community pipeline script...")
                    url = "https://raw.githubusercontent.com/huggingface/diffusers/main/examples/community/pipeline_stable_diffusion_xl_instantid.py"
                    urllib.request.urlretrieve(url, script_name)
                    print("      ✅ Script downloaded.")
                
                print("   🔧 Auto-patching community script for CPU offload bugs...")
                with open(script_name, "r", encoding="utf-8") as f:
                    script_content = f.read()
                
                bugged_line = "prompt_image_emb = self.image_proj_model(prompt_image_emb)"
                patched_line = "self.image_proj_model.to(device=device, dtype=dtype)\n        prompt_image_emb = self.image_proj_model(prompt_image_emb)"
                
                if bugged_line in script_content and patched_line not in script_content:
                    script_content = script_content.replace(bugged_line, patched_line)
                    with open(script_name, "w", encoding="utf-8") as f:
                        f.write(script_content)
                    print("      ✅ Patch applied successfully.")
                
                if os.getcwd() not in sys.path:
                    sys.path.insert(0, os.getcwd())
                    
                if 'pipeline_stable_diffusion_xl_instantid' in sys.modules:
                    del sys.modules['pipeline_stable_diffusion_xl_instantid']
                    
                from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline, draw_kps
                print("   🔄 Loading InstantID pipeline via community script...")
                
                original_check = StableDiffusionXLInstantIDPipeline.check_inputs
                def patched_check_inputs(self, *args, **kwargs):
                    try:
                        original_check(self, *args, **kwargs)
                    except Exception as e:
                        error_msg = str(e)
                        is_scale_error = "controlnet_conditioning_scale" in error_msg
                        is_type_error = "must be type" in error_msg or "subscriptable" in error_msg or "iterable" in error_msg
                        
                        if is_scale_error or (is_type_error and "float" in error_msg):
                            pass
                        else:
                            raise e
                            
                StableDiffusionXLInstantIDPipeline.check_inputs = patched_check_inputs
                
                self.pipeline = StableDiffusionXLInstantIDPipeline.from_pretrained(
                    "stabilityai/stable-diffusion-xl-base-1.0",
                    controlnet=controlnet,
                    torch_dtype=torch.float16,
                    variant="fp16",
                )
                
                self.pipeline.load_ip_adapter_instantid(face_adapter_path)
                self.pipeline.set_ip_adapter_scale(0.8)
                
                self.pipeline.scheduler = DDIMScheduler.from_config(
                    self.pipeline.scheduler.config
                )
                
                self.pipeline.enable_model_cpu_offload()
                self.pipeline_type = "instantid"
                self.available = True
                print("   ✅ InstantID pipeline loaded via community script!")
                
                self.draw_kps = draw_kps
                
            except Exception as e:
                print(f"   ⚠️ Community InstantID pipeline failed: {e}")
                self.available = False
            
        except Exception as e:
            print(f"   ⚠️  InstantID failed to load: {e}")
            import traceback
            traceback.print_exc()
            print("   📸 Falling back to traditional augmentation.")
            self.available = False
    
    def generate_variations(
        self, 
        full_image: np.ndarray,
        aligned_chip: np.ndarray,
        person_name: str
    ) -> List[Tuple[str, np.ndarray]]:
        
        if not self.available:
            print(f"   ⚠️  [DEBUG] Skipping InstantID for {person_name}: self.available is False!")
        if self.pipeline is None:
            print(f"   ⚠️  [DEBUG] Skipping InstantID for {person_name}: self.pipeline is None! (Cleanup was likely called)")
            
        if self.available and self.pipeline is not None:
            try:
                return self._generate_instantid(full_image, person_name)
            except Exception as e:
                print(f"   ⚠️  InstantID generation failed for {person_name}: {e}")
                if self.config.GGA_FALLBACK:
                    print(f"   📸 Falling back to traditional augmentation for {person_name}")
        
        return self.fallback.generate_variations(aligned_chip)
    
    def _generate_instantid(
        self, 
        full_image: np.ndarray,
        person_name: str
    ) -> List[Tuple[str, np.ndarray]]:
        
        import torch
        from PIL import Image
        
        img_rgb = cv2.cvtColor(full_image, cv2.COLOR_BGR2RGB)
        
        h, w = img_rgb.shape[:2]
        scale = 640 / min(h, w)
        if scale < 1.0:
            new_h, new_w = int(h * scale), int(w * scale)
            img_rgb = cv2.resize(img_rgb, (new_w, new_h))
            
        face_pil = Image.fromarray(img_rgb)
        
        face_info = detector.detect(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
        if face_info and len(face_info) > 0:
            face_emb = face_info[0].embedding
            face_kps = face_info[0].kps
        else:
            print(f"   ⚠️  [DEBUG] detector.detect() found NO FACES in the resized image for {person_name}!")
            face_emb = np.zeros(512, dtype=np.float32)
            face_kps = None
        
        variations = []
        
        if face_kps is not None:
            face_kps_img = self.draw_kps(face_pil, face_kps)
            
            for i, prompt in enumerate(self.config.GGA_PROMPTS):
                print(f"   🎨 Generating variation {i+1}/{len(self.config.GGA_PROMPTS)}: {prompt[:50]}...")
                
                try:
                    with torch.no_grad():
                        result = self.pipeline(
                            prompt=prompt,
                            negative_prompt="blurry, low quality, distorted, deformed",
                            image_embeds=face_emb,
                            image=face_kps_img,
                            controlnet_conditioning_scale=float(0.8),
                            num_inference_steps=25,
                            guidance_scale=5.0,
                        ).images[0]
                    
                    result_np_112 = np.array(result.resize((112, 112)))
                    result_bgr_112 = cv2.cvtColor(result_np_112, cv2.COLOR_RGB2BGR)
                    
                    variant_name = f"gen_{i}"
                    variations.append((variant_name, result_bgr_112))
                    
                    try:
                        from pathlib import Path
                        import os
                        vis_dir = Path(self.config.OUTPUT_DIR) / "gga_vis" / person_name
                        vis_dir.mkdir(parents=True, exist_ok=True)
                        
                        result_np_full = np.array(result)
                        result_bgr_full = cv2.cvtColor(result_np_full, cv2.COLOR_RGB2BGR)
                        
                        save_path = str(vis_dir / f"{variant_name}.jpg")
                        cv2.imwrite(save_path, result_bgr_full)
                        print(f"      💾 Saved cache: {save_path}")
                    except Exception as save_err:
                        print(f"      ❌ Saving cache failed: {save_err}")
                        
                except Exception as e:
                    print(f"      ⚠️ Generation {i} failed: {e}")
        
        return variations
    
    def cleanup(self):
        
        if self.pipeline is not None:
            import torch
            import gc
            
            del self.pipeline
            self.pipeline = None
            
            if hasattr(self, '_ip_adapter_state'):
                del self._ip_adapter_state
            
            gc.collect()
            
            torch.cuda.empty_cache()
            if torch.cuda.is_available():
                torch.cuda.ipc_collect()
                
            print("   🗑️  InstantID pipeline forcefully purged from GPU memory.")

augmentor = GenerativeAugmentor(config)
print("✅ Augmentor initialized!")

In [ ]:
class GalleryIndex:

    def __init__(self, config: Config):
        self.config = config
        self.dim = config.EMBEDDING_DIM
        self.index = faiss.IndexFlatIP(self.dim)
        
        self.id_map: List[Tuple[str, str]] = []
        self.person_names: set = set()
        
        print(f"   ✅ FAISS IndexFlatIP created (dim={self.dim})")
    
    def add_embedding(self, embedding: np.ndarray, person_name: str, variant: str = "original"):
        
        normalized = FaceRecognizer.normalize(embedding)
        
        vec = normalized.astype(np.float32).reshape(1, -1)
        self.index.add(vec)
        self.id_map.append((person_name, variant))
        self.person_names.add(person_name)
    
    def search(self, query_embedding: np.ndarray, k: int = 1) -> List[Tuple[str, float, str]]:
        
        normalized = FaceRecognizer.normalize(query_embedding)
        query = normalized.astype(np.float32).reshape(1, -1)
        
        similarities, indices = self.index.search(query, k)
        
        results = []
        for sim, idx in zip(similarities[0], indices[0]):
            if idx < 0 or idx >= len(self.id_map):
                continue
            person_name, variant = self.id_map[idx]
            results.append((person_name, float(sim), variant))
        
        return results
    
    def match(self, query_embedding: np.ndarray, quality_score: float = 1.0) -> Tuple[str, float]:
        
        if self.index.ntotal == 0:
            return (self.config.UNKNOWN_LABEL, 0.0)
        
        results = self.search(query_embedding, k=3)
        
        if not results:
            return (self.config.UNKNOWN_LABEL, 0.0)
        
        best_name, best_sim, best_variant = results[0]
        
        threshold = recognizer.get_dynamic_threshold(quality_score)
        
        if best_sim >= threshold:
            return (best_name, best_sim)
        else:
            return (self.config.UNKNOWN_LABEL, best_sim)
    
    def get_stats(self) -> dict:
        
        person_counts = Counter(name for name, _ in self.id_map)
        return {
            "total_vectors": self.index.ntotal,
            "total_persons": len(self.person_names),
            "persons": dict(person_counts),
        }

gallery = GalleryIndex(config)
print("✅ FAISS gallery index initialized!")

In [ ]:
class EnrollmentPipeline:

    def __init__(
        self,
        detector: FaceDetector,
        aligner: FaceAligner,
        recognizer: FaceRecognizer,
        augmentor: GenerativeAugmentor,
        gallery: GalleryIndex,
        config: Config,
    ):
        self.detector = detector
        self.aligner = aligner
        self.recognizer = recognizer
        self.augmentor = augmentor
        self.gallery = gallery
        self.config = config
    
    def enroll_person(self, image_path: str, person_name: str) -> bool:
        
        print(f"\n👤 Enrolling: {person_name}")
        print(f"   📷 Image: {image_path}")
        
        image = cv2.imread(image_path)
        if image is None:
            print(f"   ❌ Failed to load image: {image_path}")
            return False
        
        face = self.detector.detect_for_enrollment(image)
        if face is None:
            print(f"   ❌ No face detected in {image_path}")
            return False
        
        print(f"   ✅ Face detected (confidence: {face.det_score:.3f})")
        
        if face.kps is not None:
            aligned = self.aligner.align(image, face.kps)
        else:
            bbox = face.bbox.astype(int)
            crop = image[bbox[1]:bbox[3], bbox[0]:bbox[2]]
            aligned = cv2.resize(crop, (112, 112))
            print("   ⚠️  No landmarks — using crop fallback (accuracy reduced)")
        
        if vit_recognizer.available:
            kps = face.kps if face.kps is not None else None
            original_embedding = vit_recognizer.get_embedding(aligned, kps)
            if original_embedding is not None:
                print(f"   🧠 Using ViT embedding for {person_name}")
            else:
                original_embedding = face.embedding
                print(f"   📸 ViT failed, using ArcFace for {person_name}")
        else:
            original_embedding = face.embedding
        
        if original_embedding is None:
            print(f"   ❌ No embedding extracted for {person_name}")
            return False
        
        self.gallery.add_embedding(original_embedding, person_name, "original")
        print(f"   ✅ Original embedding added (dim={len(original_embedding)})")
        print(f"      📊 Embedding stats: mean={original_embedding.mean():.4f}, std={original_embedding.std():.4f}, norm={np.linalg.norm(original_embedding):.4f}")
        
        variations = self.augmentor.generate_variations(image, aligned, person_name)
        print(f"   🎨 Generated {len(variations)} variations")
        
        enrolled_variants = 0
        for variant_name, variant_chip in variations:
            var_embedding = self.detector.get_embedding_direct(variant_chip)
            if var_embedding is not None:
                self.gallery.add_embedding(var_embedding, person_name, variant_name)
                enrolled_variants += 1
            else:
                padded = cv2.copyMakeBorder(variant_chip, 40, 40, 40, 40, 
                                           cv2.BORDER_REPLICATE)
                var_faces = self.detector.detect(padded)
                if var_faces:
                    best_var = max(var_faces, key=lambda f: f.det_score)
                    if best_var.kps is not None:
                        var_aligned = self.aligner.align(padded, best_var.kps)
                        var_emb = vit_recognizer.get_embedding(var_aligned) if vit_recognizer.available else best_var.embedding
                    else:
                        var_emb = best_var.embedding
                    if var_emb is not None:
                        self.gallery.add_embedding(var_emb, person_name, variant_name)
                        enrolled_variants += 1
        
        print(f"   ✅ Enrolled {enrolled_variants + 1} vectors total for {person_name}")
        
        for scale_name, scale in [("zoomed_in", 1.2), ("zoomed_out", 0.8)]:
            h, w = image.shape[:2]
            scaled = cv2.resize(image, (int(w*scale), int(h*scale)))
            scaled_faces = self.detector.detect(scaled)
            if scaled_faces:
                best_face = max(scaled_faces, key=lambda f: f.det_score)
                if best_face.kps is not None:
                    scaled_aligned = self.aligner.align(scaled, best_face.kps)
                    if vit_recognizer.available:
                        scaled_emb = vit_recognizer.get_embedding(scaled_aligned)
                    else:
                        scaled_emb = best_face.embedding
                    if scaled_emb is not None:
                        self.gallery.add_embedding(scaled_emb, person_name, scale_name)
                        enrolled_variants += 1
        
        print(f"   ✅ Final total: {enrolled_variants + 1} vectors for {person_name}")
        return True
    
    def enroll_directory(self, directory: str) -> dict:
        
        print("\n" + "="*60)
        print("📋 ENROLLMENT PIPELINE")
        print("="*60)
        
        directory = Path(directory)
        if not directory.exists():
            print(f"❌ Directory not found: {directory}")
            return {"success": 0, "failed": 0, "total": 0}
        
        image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        image_files = [
            f for f in sorted(directory.iterdir()) 
            if f.suffix.lower() in image_extensions
        ]
        
        print(f"📁 Found {len(image_files)} images in {directory}")
        
        results = {"success": 0, "failed": 0, "total": len(image_files), "persons": []}
        
        for img_file in image_files:
            person_name = img_file.stem
            success = self.enroll_person(str(img_file), person_name)
            
            if success:
                results["success"] += 1
                results["persons"].append(person_name)
            else:
                results["failed"] += 1
        
        self.augmentor.cleanup()
        
        print("\n" + "="*60)
        print(f"📊 ENROLLMENT COMPLETE")
        print(f"   ✅ Success: {results['success']}/{results['total']}")
        print(f"   ❌ Failed:  {results['failed']}/{results['total']}")
        print(f"   📊 Gallery: {self.gallery.get_stats()}")
        print("="*60)
        
        return results

enrollment = EnrollmentPipeline(detector, aligner, recognizer, augmentor, gallery, config)
print("✅ Enrollment pipeline ready!")

enrollment_results = enrollment.enroll_directory(config.ENROLLMENT_DIR)

In [ ]:
import matplotlib.pyplot as plt

def visualize_gga_results(person_name: str, config: Config):
    
    if not config.ENABLE_GGA:
        print("⚠️ GGA is not enabled. Only traditional variants exist.")
        return
        
    print(f"\n🔍 Visualizing variations for: {person_name}")
    
    from pathlib import Path
    import matplotlib.pyplot as plt
    
    vis_dir = Path(config.OUTPUT_DIR) / "gga_vis" / person_name
    if not vis_dir.exists():
        print(f"❌ No cached generative images found for {person_name} in {vis_dir}")
        print("   Make sure you run the Enrollment Cell 8 first so they are saved!")
        return
        
    enrollment_dir = Path(config.ENROLLMENT_DIR)
    orig_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = enrollment_dir / f"{person_name}{ext}"
        if candidate.exists():
            orig_path = candidate
            break
            
    variant_files = list(vis_dir.glob("gen_*.jpg"))
    if not variant_files:
        print(f"❌ No variant images found in {vis_dir}")
        return
        
    num_images = len(variant_files) + (1 if orig_path else 0)
    fig, axes = plt.subplots(1, num_images, figsize=(4 * num_images, 4))
    
    if num_images == 1:
        axes = [axes]
        
    idx = 0
    if orig_path:
        orig_img = plt.imread(str(orig_path))
        axes[idx].imshow(orig_img)
        axes[idx].set_title("Original")
        axes[idx].axis('off')
        idx += 1
        
    for var_file in sorted(variant_files):
        var_img = plt.imread(str(var_file))
        axes[idx].imshow(var_img)
        axes[idx].set_title(var_file.stem)
        axes[idx].axis('off')
        idx += 1
        
    plt.tight_layout()
    plt.show()

visualize_gga_results("aaditya_singhal", config)

In [ ]:
class IdentityTracker:

    def __init__(self, config: Config):
        self.config = config
        self.tracker = sv.ByteTrack(
            track_activation_threshold=config.TRACK_THRESH,
            lost_track_buffer=config.TRACK_BUFFER,
            minimum_matching_threshold=config.MATCH_THRESH,
            frame_rate=30,
        )
        
        self.identity_votes: Dict[int, List[str]] = defaultdict(list)
        self.confirmed_identities: Dict[int, str] = {}
        
        print("   ✅ ByteTrack tracker initialized")
    
    def update(self, detections: sv.Detections) -> sv.Detections:
        
        return self.tracker.update_with_detections(detections)
    
    def vote_identity(self, track_id: int, identity: str) -> str:
        
        votes = self.identity_votes[track_id]
        votes.append(identity)
        
        if len(votes) > self.config.VOTE_WINDOW:
            votes = votes[-self.config.VOTE_WINDOW:]
            self.identity_votes[track_id] = votes
        
        known_votes = [v for v in votes if v != self.config.UNKNOWN_LABEL]
        
        if known_votes:
            counter = Counter(known_votes)
            best_name, best_count = counter.most_common(1)[0]
            
            if best_count >= self.config.VOTE_MIN:
                self.confirmed_identities[track_id] = best_name
                return best_name
        
        if track_id in self.confirmed_identities:
            return self.confirmed_identities[track_id]
        
        return self.config.UNKNOWN_LABEL
    
    def get_identity(self, track_id: int) -> str:
        
        return self.confirmed_identities.get(track_id, self.config.UNKNOWN_LABEL)
    
    def get_all_confirmed(self) -> Dict[int, str]:
        
        return dict(self.confirmed_identities)

tracker = IdentityTracker(config)
print("✅ Identity tracker initialized!")

In [ ]:
class AttendanceInference:

    def __init__(
        self,
        detector: FaceDetector,
        aligner: FaceAligner,
        recognizer: FaceRecognizer,
        gallery: GalleryIndex,
        tracker: IdentityTracker,
        config: Config,
    ):
        self.detector = detector
        self.aligner = aligner
        self.recognizer = recognizer
        self.gallery = gallery
        self.tracker = tracker
        self.config = config
        
        self.attendance: Dict[str, dict] = {}
        self.frame_count = 0
    
    def _faces_to_detections(self, faces: list) -> sv.Detections:
        
        if not faces:
            return sv.Detections.empty()
        
        bboxes = np.array([f.bbox for f in faces])
        confidences = np.array([f.det_score for f in faces])
        
        return sv.Detections(
            xyxy=bboxes,
            confidence=confidences,
        )
    
    def process_frame(self, frame: np.ndarray, do_recognize: bool = True) -> Tuple[np.ndarray, list]:
        
        self.frame_count += 1
        
        faces = self.detector.detect(frame)
        
        detections = self._faces_to_detections(faces)
        tracked = self.tracker.update(detections)
        
        frame_results = []
        
        if do_recognize and tracked.tracker_id is not None:
            for i, (bbox, track_id) in enumerate(zip(tracked.xyxy, tracked.tracker_id)):
                face = self._find_matching_face(faces, bbox)
                
                if face is not None:
                    embedding = None
                    if vit_recognizer.available and face.kps is not None:
                        aligned_chip = self.aligner.align(frame, face.kps)
                        embedding = vit_recognizer.get_embedding(aligned_chip, face.kps)
                    
                    if embedding is None:
                        embedding = face.embedding
                    
                    if embedding is not None:
                        quality = self.recognizer.compute_quality_score(face)
                        
                        identity, similarity = self.gallery.match(embedding, quality)
                        
                        confirmed = self.tracker.vote_identity(track_id, identity)
                        
                        frame_results.append({
                            "track_id": int(track_id),
                            "identity": confirmed,
                            "similarity": similarity,
                            "quality": quality,
                            "bbox": bbox.tolist(),
                        })
                        
                        if confirmed != self.config.UNKNOWN_LABEL:
                            self._update_attendance(confirmed)
        
        annotated = self._annotate_frame(frame, tracked)
        
        return annotated, frame_results
    
    def _find_matching_face(self, faces: list, bbox: np.ndarray, iou_thresh: float = 0.5):
        
        best_face = None
        best_iou = 0
        
        for face in faces:
            iou = self._compute_iou(face.bbox, bbox)
            if iou > best_iou:
                best_iou = iou
                best_face = face
        
        return best_face if best_iou >= iou_thresh else None
    
    @staticmethod
    def _compute_iou(box1: np.ndarray, box2: np.ndarray) -> float:
        
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        intersection = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - intersection
        
        return intersection / union if union > 0 else 0
    
    def _update_attendance(self, person_name: str):
        
        now = self.frame_count
        if person_name not in self.attendance:
            self.attendance[person_name] = {
                "first_seen_frame": now,
                "last_seen_frame": now,
                "detection_count": 1,
                "status": "PRESENT",
            }
        else:
            self.attendance[person_name]["last_seen_frame"] = now
            self.attendance[person_name]["detection_count"] += 1
    
    def _annotate_frame(self, frame: np.ndarray, tracked: sv.Detections) -> np.ndarray:
        
        annotated = frame.copy()
        
        if tracked.tracker_id is None:
            return annotated
        
        for bbox, track_id in zip(tracked.xyxy, tracked.tracker_id):
            identity = self.tracker.get_identity(track_id)
            bbox_int = bbox.astype(int)
            
            if identity == self.config.UNKNOWN_LABEL:
                color = self.config.UNKNOWN_COLOR
                label = f"Unknown (#{track_id})"
            else:
                color = self.config.BBOX_COLOR
                label = f"{identity} (#{track_id})"
            
            cv2.rectangle(annotated, 
                         (bbox_int[0], bbox_int[1]), 
                         (bbox_int[2], bbox_int[3]),
                         color, 2)
            
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 
                                           self.config.FONT_SCALE, self.config.FONT_THICKNESS)
            cv2.rectangle(annotated,
                         (bbox_int[0], bbox_int[1] - th - 10),
                         (bbox_int[0] + tw, bbox_int[1]),
                         color, -1)
            
            cv2.putText(annotated, label,
                       (bbox_int[0], bbox_int[1] - 5),
                       cv2.FONT_HERSHEY_SIMPLEX,
                       self.config.FONT_SCALE,
                       (255, 255, 255),
                       self.config.FONT_THICKNESS)
        
        return annotated
    
    def run_on_video(self, video_path: str) -> dict:
        
        print("\n" + "="*60)
        print("🎥 VIDEO INFERENCE PIPELINE")
        print("="*60)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"❌ Failed to open video: {video_path}")
            return {}
        
        fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        print(f"   📐 Resolution: {width}×{height}")
        print(f"   🎬 FPS: {fps}, Total frames: {total_frames}")
        print(f"   ⏩ Processing every {self.config.PROCESS_EVERY_N_FRAMES}th frame")
        print(f"   🧠 Recognition every {self.config.RECOGNIZE_EVERY_N_FRAMES}th frame")
        
        out_path = os.path.join(self.config.OUTPUT_DIR, "annotated_output.mp4")
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out_fps = fps // self.config.PROCESS_EVERY_N_FRAMES
        writer = cv2.VideoWriter(out_path, fourcc, out_fps, (width, height))
        
        frame_idx = 0
        processed = 0
        all_results = []
        start_time = time.time()
        
        frames_to_process = min(total_frames, self.config.MAX_FRAMES)
        
        while True:
            ret, frame = cap.read()
            if not ret or frame_idx >= frames_to_process:
                break
            
            frame_idx += 1
            
            if frame_idx % self.config.PROCESS_EVERY_N_FRAMES != 0:
                continue
            
            do_recognize = (processed % (self.config.RECOGNIZE_EVERY_N_FRAMES // 
                                         self.config.PROCESS_EVERY_N_FRAMES) == 0)
            
            annotated, results = self.process_frame(frame, do_recognize=do_recognize)
            writer.write(annotated)
            all_results.extend(results)
            processed += 1
            
            if processed % 10 == 0:
                elapsed = time.time() - start_time
                fps_actual = processed / elapsed if elapsed > 0 else 0
                progress = frame_idx / frames_to_process * 100
                print(f"   ⏳ Progress: {progress:.0f}% | Frame {frame_idx}/{frames_to_process} | "
                      f"{fps_actual:.1f} FPS | Faces tracked: {len(self.tracker.get_all_confirmed())}")
        
        cap.release()
        writer.release()
        
        elapsed = time.time() - start_time
        
        self._save_sample_frames(video_path, frames_to_process)
        
        print(f"\n   ✅ Processing complete!")
        print(f"   ⏱️  Time: {elapsed:.1f}s ({processed/elapsed:.1f} processed FPS)")
        print(f"   📹 Output video: {out_path}")
        
        return {
            "output_video": out_path,
            "total_frames": frame_idx,
            "processed_frames": processed,
            "elapsed_seconds": elapsed,
            "fps": processed / elapsed if elapsed > 0 else 0,
            "attendance": self.attendance,
            "all_detections": len(all_results),
        }
    
    def _save_sample_frames(self, video_path: str, total_frames: int, num_samples: int = 5):
        
        cap = cv2.VideoCapture(video_path)
        sample_indices = np.linspace(0, total_frames - 1, num_samples, dtype=int)
        
        for i, target_frame in enumerate(sample_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
            ret, frame = cap.read()
            if ret:
                faces = self.detector.detect(frame)
                annotated = self.detector.visualize(frame, faces)
                
                cv2.putText(annotated, f"Frame: {target_frame}", (10, 30),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
                
                out_path = os.path.join(self.config.OUTPUT_DIR, f"sample_frame_{i}.jpg")
                cv2.imwrite(out_path, annotated)
        
        cap.release()
        print(f"   📸 Saved {num_samples} sample frames to {self.config.OUTPUT_DIR}")
    
    def get_attendance_report(self) -> str:
        
        report = []
        report.append("\n" + "="*60)
        report.append("📋 ATTENDANCE REPORT")
        report.append("="*60)
        
        if not self.attendance:
            report.append("   No students detected.")
            return "\n".join(report)
        
        report.append(f"\n{'Name':<20} {'Status':<10} {'Detections':<12} {'First Frame':<12} {'Last Frame':<12}")
        report.append("-" * 66)
        
        for name, data in sorted(self.attendance.items()):
            report.append(
                f"{name:<20} {data['status']:<10} {data['detection_count']:<12} "
                f"{data['first_seen_frame']:<12} {data['last_seen_frame']:<12}"
            )
        
        report.append("-" * 66)
        report.append(f"Total present: {len(self.attendance)}")
        
        enrolled = gallery.person_names
        present = set(self.attendance.keys())
        absent = enrolled - present
        
        if absent:
            report.append(f"\n⚠️  ABSENT ({len(absent)}):")
            for name in sorted(absent):
                report.append(f"   - {name}")
        
        report.append("="*60)
        return "\n".join(report)

inference = AttendanceInference(detector, aligner, recognizer, gallery, tracker, config)
print("✅ Inference pipeline ready!")

In [ ]:
import re

GDRIVE_LINK = "https://drive.google.com/file/d/1dxS1TROoTp_SwYQ0iXsWU7EIfMY5BXyP/view?usp=sharing"

def download_from_gdrive(link: str, output_dir: str) -> str:
    
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown
    
    file_id = None
    
    match = re.search(r'/file/d/([a-zA-Z0-9_-]+)', link)
    if match:
        file_id = match.group(1)
    
    if not file_id:
        match = re.search(r'[?&]id=([a-zA-Z0-9_-]+)', link)
        if match:
            file_id = match.group(1)
    
    if not file_id:
        file_id = link.strip()
    
    print(f"📥 Downloading video from Google Drive...")
    print(f"   File ID: {file_id}")
    
    output_path = os.path.join(output_dir, "gdrive_video.mp4")
    url = f"https://drive.google.com/uc?id={file_id}"
    
    gdown.download(url, output_path, quiet=False)
    
    if os.path.exists(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"   ✅ Downloaded: {output_path} ({size_mb:.1f} MB)")
    else:
        print(f"   ❌ Download failed!")
        
    return output_path

os.makedirs(config.OUTPUT_DIR, exist_ok=True)
video_path = download_from_gdrive(GDRIVE_LINK, config.OUTPUT_DIR)

if os.path.exists(video_path):
    tracker_v = IdentityTracker(config)
    inference_v = AttendanceInference(
        detector, aligner, recognizer, gallery, tracker_v, config
    )
    
    print(f"\n{'='*60}")
    print(f"🎬 Processing: {os.path.basename(video_path)}")
    print(f"{'='*60}")
    
    results = inference_v.run_on_video(video_path)
    
    report = inference_v.get_attendance_report()
    print(report)
    
    attendance_path = os.path.join(config.OUTPUT_DIR, "attendance.json")
    with open(attendance_path, "w") as f:
        json.dump({
            "video": os.path.basename(video_path),
            "attendance": inference_v.attendance,
            "gallery_stats": gallery.get_stats(),
            "inference_stats": {
                "total_frames": results.get("total_frames", 0),
                "processed_frames": results.get("processed_frames", 0),
                "elapsed_seconds": results.get("elapsed_seconds", 0),
                "fps": results.get("fps", 0),
            }
        }, f, indent=2)
    
    print(f"\n💾 Attendance saved to: {attendance_path}")
else:
    print("❌ Video file not found. Check your Google Drive link and sharing permissions.")

In [ ]:
print("\n" + "="*60)
print("📏 BENCHMARKING & DIAGNOSTICS")
print("="*60)

print("\n1️⃣  Gallery Embedding Quality")
print("-"*40)

if gallery.index.ntotal > 0:
    all_vectors = faiss.rev_swig_ptr(gallery.index.get_xb(), gallery.index.ntotal * 512)
    all_vectors = np.array(all_vectors).reshape(gallery.index.ntotal, 512)
    all_labels = [name for name, _ in gallery.id_map]
    
    unique_names = sorted(set(all_labels))
    centroids = {}
    for name in unique_names:
        idxs = [i for i, l in enumerate(all_labels) if l == name]
        person_vecs = all_vectors[idxs]
        centroid = person_vecs.mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-8)
        centroids[name] = centroid
        
        intra_sims = person_vecs @ centroid
        print(f"   {name:20s}: {len(idxs):2d} vectors | "
              f"intra-sim: {intra_sims.mean():.4f} ± {intra_sims.std():.4f} | "
              f"min: {intra_sims.min():.4f}")
    
    print(f"\n   📊 Inter-Class Centroid Similarity Matrix:")
    print(f"   {'':20s}", end="")
    for n in unique_names:
        print(f" {n[:8]:>8s}", end="")
    print()
    
    for n1 in unique_names:
        print(f"   {n1:20s}", end="")
        for n2 in unique_names:
            sim = float(centroids[n1] @ centroids[n2])
            marker = "██" if n1 == n2 else ("⚠️" if sim > 0.5 else "  ")
            print(f" {sim:8.4f}", end="")
        print()
    
    inter_sims = []
    for i, n1 in enumerate(unique_names):
        for n2 in unique_names[i+1:]:
            inter_sims.append(float(centroids[n1] @ centroids[n2]))
    
    if inter_sims:
        print(f"\n   ✅ Inter-class similarity: mean={np.mean(inter_sims):.4f}, "
              f"max={np.max(inter_sims):.4f}")
        print(f"   {'👍 GOOD' if np.max(inter_sims) < 0.5 else '⚠️  WARNING: Some inter-class sims > 0.5'}")

print(f"\n2️⃣  Inference Performance")
print("-"*40)

if vit_recognizer.available:
    import torch
    dummy_face = np.random.randint(0, 255, (112, 112, 3), dtype=np.uint8)
    
    for _ in range(3):
        vit_recognizer.get_embedding(dummy_face)
    
    n_runs = 20
    t0 = time.time()
    for _ in range(n_runs):
        vit_recognizer.get_embedding(dummy_face)
    vit_time = (time.time() - t0) / n_runs * 1000
    print(f"   ViT embedding:   {vit_time:.1f} ms/face ({1000/vit_time:.0f} faces/sec)")

dummy_face_arc = np.random.randint(0, 255, (112, 112, 3), dtype=np.uint8)
t0 = time.time()
for _ in range(20):
    detector._original_unpatched_get_embedding_direct(dummy_face_arc)
arc_time = (time.time() - t0) / 20 * 1000
print(f"   ArcFace embedding: {arc_time:.1f} ms/face ({1000/arc_time:.0f} faces/sec)")

dummy_frame = np.random.randint(0, 255, (480, 848, 3), dtype=np.uint8)
t0 = time.time()
for _ in range(10):
    detector.detect(dummy_frame)
det_time = (time.time() - t0) / 10 * 1000
print(f"   SCRFD detection:  {det_time:.1f} ms/frame")

print(f"\n4️⃣  Rank-1 Identification Accuracy")
print("-"*40)

if gallery.index.ntotal > 0:
    correct = 0
    total = 0
    per_person_correct = {n: 0 for n in unique_names}
    per_person_total = {n: 0 for n in unique_names}
    
    for i in range(gallery.index.ntotal):
        query = all_vectors[i:i+1].copy()
        true_label = all_labels[i]
        
        D, I = gallery.index.search(query, k=2)
        
        for j in range(2):
            idx = int(I[0][j])
            if idx != i:
                pred_label = all_labels[idx]
                if pred_label == true_label:
                    correct += 1
                    per_person_correct[true_label] += 1
                break
        total += 1
        per_person_total[true_label] += 1
    
    rank1_acc = correct / total * 100
    print(f"   Overall Rank-1 Accuracy: {rank1_acc:.1f}% ({correct}/{total})")
    print(f"\n   Per-person breakdown:")
    for name in unique_names:
        acc = per_person_correct[name] / per_person_total[name] * 100 if per_person_total[name] > 0 else 0
        bar = "█" * int(acc / 5) + "░" * (20 - int(acc / 5))
        print(f"   {name:20s}: {acc:5.1f}% {bar}")

print(f"\n5️⃣  Threshold Sensitivity Analysis")
print("-"*40)

if gallery.index.ntotal > 0:
    genuine_scores = []
    impostor_scores = []
    
    for i in range(gallery.index.ntotal):
        for j in range(i + 1, min(i + 20, gallery.index.ntotal)):
            sim = float(all_vectors[i] @ all_vectors[j])
            if all_labels[i] == all_labels[j]:
                genuine_scores.append(sim)
            else:
                impostor_scores.append(sim)
    
    genuine_scores = np.array(genuine_scores) if genuine_scores else np.array([0.0])
    impostor_scores = np.array(impostor_scores) if impostor_scores else np.array([0.0])
    
    print(f"   Genuine pairs:   {len(genuine_scores):4d} | mean={genuine_scores.mean():.4f} | "
          f"min={genuine_scores.min():.4f} | max={genuine_scores.max():.4f}")
    print(f"   Impostor pairs:  {len(impostor_scores):4d} | mean={impostor_scores.mean():.4f} | "
          f"min={impostor_scores.min():.4f} | max={impostor_scores.max():.4f}")
    
    thresholds = [0.2, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7]
    print(f"\n   {'Threshold':>10s} {'TPR':>8s} {'FPR':>8s} {'Precision':>10s} {'F1':>8s}")
    print(f"   {'─'*10} {'─'*8} {'─'*8} {'─'*10} {'─'*8}")
    
    best_f1 = 0
    best_thresh = 0.4
    for t in thresholds:
        tp = np.sum(genuine_scores >= t)
        fn = np.sum(genuine_scores < t)
        fp = np.sum(impostor_scores >= t)
        tn = np.sum(impostor_scores < t)
        
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0
        
        marker = " ◀── current" if abs(t - 0.4) < 0.01 else ""
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
        
        print(f"   {t:10.2f} {tpr:8.4f} {fpr:8.4f} {precision:10.4f} {f1:8.4f}{marker}")
    
    print(f"\n   🏆 Best threshold: {best_thresh:.2f} (F1={best_f1:.4f})")

print(f"\n6️⃣  ROC Curve & AUC")
print("-"*40)

if gallery.index.ntotal > 0 and len(genuine_scores) > 0 and len(impostor_scores) > 0:
    all_scores = np.concatenate([genuine_scores, impostor_scores])
    all_truth = np.concatenate([np.ones(len(genuine_scores)), np.zeros(len(impostor_scores))])
    
    roc_thresholds = np.linspace(all_scores.min() - 0.01, all_scores.max() + 0.01, 200)
    tpr_list = []
    fpr_list = []
    
    for t in roc_thresholds:
        tp = np.sum(genuine_scores >= t)
        fn = np.sum(genuine_scores < t)
        fp = np.sum(impostor_scores >= t)
        tn = np.sum(impostor_scores < t)
        
        tpr_list.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    
    tpr_arr = np.array(tpr_list)
    fpr_arr = np.array(fpr_list)
    
    sorted_idx = np.argsort(fpr_arr)
    fpr_sorted = fpr_arr[sorted_idx]
    tpr_sorted = tpr_arr[sorted_idx]
    auc = np.trapz(tpr_sorted, fpr_sorted)
    
    print(f"   AUC (Area Under ROC Curve): {auc:.4f}")
    print(f"   {'🏆 Excellent' if auc > 0.99 else '✅ Good' if auc > 0.95 else '⚠️ Needs improvement'}")
    
    print(f"\n   Key operating points:")
    for target_fpr in [0.001, 0.01, 0.1]:
        idx = np.argmin(np.abs(fpr_arr - target_fpr))
        print(f"   FPR={target_fpr:.3f} → TPR={tpr_arr[idx]:.4f} (threshold={roc_thresholds[idx]:.3f})")
    
    fnr_arr = 1 - tpr_arr
    eer_idx = np.argmin(np.abs(fpr_arr - fnr_arr))
    eer = (fpr_arr[eer_idx] + fnr_arr[eer_idx]) / 2
    print(f"\n   EER (Equal Error Rate): {eer:.4f} ({eer*100:.2f}%)")
    print(f"   {'🏆 Excellent' if eer < 0.01 else '✅ Good' if eer < 0.05 else '⚠️ Needs improvement'}")

print(f"\n7️⃣  Silhouette Score (Clustering Quality — R² Equivalent)")
print("-"*40)

if gallery.index.ntotal > 0 and len(unique_names) > 1:
    
    silhouette_scores = []
    for i in range(gallery.index.ntotal):
        label_i = all_labels[i]
        vec_i = all_vectors[i]
        
        same_idxs = [j for j in range(gallery.index.ntotal) if all_labels[j] == label_i and j != i]
        if not same_idxs:
            continue
        same_vecs = all_vectors[same_idxs]
        a_i = 1.0 - float(np.mean(vec_i @ same_vecs.T))
        
        b_i = float('inf')
        for other_name in unique_names:
            if other_name == label_i:
                continue
            other_idxs = [j for j in range(gallery.index.ntotal) if all_labels[j] == other_name]
            other_vecs = all_vectors[other_idxs]
            mean_dist = 1.0 - float(np.mean(vec_i @ other_vecs.T))
            b_i = min(b_i, mean_dist)
        
        s_i = (b_i - a_i) / max(a_i, b_i) if max(a_i, b_i) > 0 else 0
        silhouette_scores.append(s_i)
    
    mean_silhouette = np.mean(silhouette_scores)
    print(f"   Mean Silhouette Score: {mean_silhouette:.4f}")
    print(f"   Interpretation:")
    print(f"     1.0  = Perfect separation (every embedding perfectly clustered)")
    print(f"     0.5+ = Good separation (clear identity clusters)")
    print(f"     0.0  = Overlapping clusters (confusion likely)")
    print(f"    -1.0  = Wrong clustering (embeddings assigned to wrong identity)")
    quality = ("🏆 Excellent" if mean_silhouette > 0.7 else 
               "✅ Good" if mean_silhouette > 0.5 else 
               "⚠️ Moderate" if mean_silhouette > 0.25 else "❌ Poor")
    print(f"   Result: {quality}")
    
    print(f"\n   📝 Note on R² Score:")
    print(f"   R² is a regression metric (predicting continuous values).")
    print(f"   For classification/recognition, these are the standard equivalents:")
    print(f"     • Silhouette Score → measures cluster quality (like R² for clusters)")
    print(f"     • AUC-ROC         → measures discriminative ability")
    print(f"     • EER             → single-number error rate (lower = better)")
    print(f"     • Rank-1 Accuracy → % of correct top-1 identification")

print(f"\n8️⃣  Pipeline Summary")
print("-"*40)
print(f"   Gallery size:     {gallery.index.ntotal} vectors")
print(f"   Gallery persons:  {len(set(all_labels)) if gallery.index.ntotal > 0 else 0}")
print(f"   Recognition model: {'AdaFace ViT-Base KP-RPE' if vit_recognizer.available else 'ArcFace ResNet'}")
print(f"   Detection model:  SCRFD-10G")
print(f"   Tracker:          ByteTrack")
print(f"   Index type:       FAISS FlatIP (exact search)")

print("\n" + "="*60)
print("📏 BENCHMARKING COMPLETE")
print("="*60)